### GBFS real-time ingestion (US06)

**User Story:** US06 – Connect to GBFS real-time endpoints  
**Objective:**  
Retrieve real-time station information and status data from the official Bike Share Toronto GBFS feeds.
**Output:**  
Spark DataFrames with extraction timestamp, ready for validation and Bronze persistence.

# Task – Identify and validate GBFS real-time endpoints
This section identifies the official Bike Share Toronto GBFS feed and validates the availability of the required real-time endpoints.

In [0]:
import requests
import json

# Official GBFS entry point for Bike Share Toronto
gbfs_root_url = "https://toronto-us.publicbikesystem.net/customer/gbfs/v2/gbfs.json"

response = requests.get(gbfs_root_url)
response.raise_for_status()

gbfs_root = response.json()

gbfs_root.keys()

dict_keys(['last_updated', 'ttl', 'data', 'version'])

In [0]:
# Inspect available languages
gbfs_root["data"].keys()

dict_keys(['en', 'fr', 'es'])

In [0]:
# Select English feeds
en_feeds = gbfs_root["data"]["en"]["feeds"]
en_feeds


[{'name': 'geofencing_zones',
  'url': 'https://toronto.publicbikesystem.net/customer/gbfs/v2/en/geofencing_zones'},
 {'name': 'gbfs_versions',
  'url': 'https://toronto.publicbikesystem.net/customer/gbfs/v2/gbfs_versions'},
 {'name': 'vehicle_types',
  'url': 'https://toronto.publicbikesystem.net/customer/gbfs/v2/en/vehicle_types'},
 {'name': 'station_information',
  'url': 'https://toronto.publicbikesystem.net/customer/gbfs/v2/en/station_information'},
 {'name': 'station_status',
  'url': 'https://toronto.publicbikesystem.net/customer/gbfs/v2/en/station_status'},
 {'name': 'system_regions',
  'url': 'https://toronto.publicbikesystem.net/customer/gbfs/v2/en/system_regions'},
 {'name': 'system_information',
  'url': 'https://toronto.publicbikesystem.net/customer/gbfs/v2/en/system_information'},
 {'name': 'system_pricing_plans',
  'url': 'https://toronto.publicbikesystem.net/customer/gbfs/v2/en/system_pricing_plans'}]

In [0]:
# Convert feeds list to a dictionary for easy access
feeds_dict = {feed["name"]: feed["url"] for feed in en_feeds}

feeds_dict


{'geofencing_zones': 'https://toronto.publicbikesystem.net/customer/gbfs/v2/en/geofencing_zones',
 'gbfs_versions': 'https://toronto.publicbikesystem.net/customer/gbfs/v2/gbfs_versions',
 'vehicle_types': 'https://toronto.publicbikesystem.net/customer/gbfs/v2/en/vehicle_types',
 'station_information': 'https://toronto.publicbikesystem.net/customer/gbfs/v2/en/station_information',
 'station_status': 'https://toronto.publicbikesystem.net/customer/gbfs/v2/en/station_status',
 'system_regions': 'https://toronto.publicbikesystem.net/customer/gbfs/v2/en/system_regions',
 'system_information': 'https://toronto.publicbikesystem.net/customer/gbfs/v2/en/system_information',
 'system_pricing_plans': 'https://toronto.publicbikesystem.net/customer/gbfs/v2/en/system_pricing_plans'}

In [0]:
required_feeds = ["station_information", "station_status"]

for feed in required_feeds:
    url = feeds_dict.get(feed)
    resp = requests.get(url)
    print(feed, resp.status_code)

station_information 200
station_status 200


# Task – Extract GBFS real-time data from selected endpoints
This section retrieves real-time data from the selected GBFS endpoints and returns Spark DataFrames for on-demand inspection and validation.

In [0]:
import requests
import pandas as pd
from pyspark.sql.functions import current_timestamp, lit

# Note: GBFS data is queried on demand and not persisted or integrated with historical datasets.

def fetch_gbfs_json(url: str) -> dict:
    resp = requests.get(url)
    resp.raise_for_status()
    return resp.json()

def gbfs_stations_to_spark_df(stations: list, endpoint_name: str):
    pdf = pd.json_normalize(stations)
    return (
        spark.createDataFrame(pdf)
             .withColumn("retrieval_timestamp", current_timestamp())
             .withColumn("source_endpoint", lit(endpoint_name))
    )

def get_station_information_df(feeds_dict: dict):
    data = fetch_gbfs_json(feeds_dict["station_information"])["data"]["stations"]
    return gbfs_stations_to_spark_df(data, "station_information")

def get_station_status_df(feeds_dict: dict):
    data = fetch_gbfs_json(feeds_dict["station_status"])["data"]["stations"]
    return gbfs_stations_to_spark_df(data, "station_status")

In [0]:
df_station_info = get_station_information_df(feeds_dict)
df_station_status = get_station_status_df(feeds_dict)


In [0]:
df_station_info.show(5, truncate=False)
df_station_status.show(5, truncate=False)

+----------+----------------------------+----------------------+-----------------+------------------+----------------------------+--------+-------------------+------------------+-------------------------------------+------------------+--------------------------------------------------+------------+------------+---------------+-------------+------------------+---------+--------+----------------+------------+--------------------------+-------------------+
|station_id|name                        |physical_configuration|lat              |lon               |address                     |capacity|is_charging_station|geofenced_capacity|rental_methods                       |is_virtual_station|groups                                            |obcn        |short_name  |nearby_distance|_bluetooth_id|_ride_code_support|post_code|altitude|is_valet_station|cross_street|retrieval_timestamp       |source_endpoint    |
+----------+----------------------------+----------------------+-----------------+--

In [0]:
df_station_info.printSchema()
df_station_status.printSchema()

root
 |-- station_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- physical_configuration: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- address: string (nullable = true)
 |-- capacity: long (nullable = true)
 |-- is_charging_station: boolean (nullable = true)
 |-- geofenced_capacity: long (nullable = true)
 |-- rental_methods: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- is_virtual_station: boolean (nullable = true)
 |-- groups: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- obcn: string (nullable = true)
 |-- short_name: string (nullable = true)
 |-- nearby_distance: double (nullable = true)
 |-- _bluetooth_id: string (nullable = true)
 |-- _ride_code_support: boolean (nullable = true)
 |-- post_code: string (nullable = true)
 |-- altitude: double (nullable = true)
 |-- is_valet_station: boolean (nullable = true)
 |-- cross_street: string (nullable 

### Task 33 – Validate GBFS data structure and ingestion integrity
This section performs basic validation checks to confirm that  the real-time GBFS data was retrieved successfully and that key fields are present.

In [0]:
print("station_information rows:", df_station_info.count())
print("station_status rows:", df_station_status.count())
required_info_cols = ["station_id", "name", "lat", "lon", "capacity", "retrieval_timestamp", "source_endpoint"]
required_status_cols = ["station_id", "num_bikes_available", "num_docks_available", "last_reported", "retrieval_timestamp", "source_endpoint"]

missing_info = [c for c in required_info_cols if c not in df_station_info.columns]
missing_status = [c for c in required_status_cols if c not in df_station_status.columns]

print("Missing station_information columns:", missing_info)
print("Missing station_status columns:", missing_status)

station_information rows: 1007
station_status rows: 1007
Missing station_information columns: []
Missing station_status columns: []


In [0]:
from pyspark.sql.functions import col

# Ensure retrieval_timestamp exists and is not null
print("Null retrieval_timestamp (info):", df_station_info.filter(col("retrieval_timestamp").isNull()).count())
print("Null retrieval_timestamp (status):", df_station_status.filter(col("retrieval_timestamp").isNull()).count())

# Optional: check last_reported presence (may be null for some rows depending on feed)
print("Null last_reported (status):", df_station_status.filter(col("last_reported").isNull()).count())

Null retrieval_timestamp (info): 0
Null retrieval_timestamp (status): 0
Null last_reported (status): 0


### Task – Document GBFS ingestion outcome and usage

**Data source**
- Bike Share Toronto – GBFS real-time feeds

**Scope**
- This module provides on-demand access to real-time station data.
- GBFS data is not persisted and is not integrated with historical datasets (2023–2024).

**Available endpoints**
- station_information: static metadata for each station.
- station_status: real-time operational availability.

**How to use**
- Call `get_station_information_df(feeds_dict)` to retrieve station metadata.
- Call `get_station_status_df(feeds_dict)` to retrieve real-time availability.
- Each call returns a Spark DataFrame with `retrieval_timestamp`.

**Limitations**
- GBFS reflects current conditions only.
- Data is suitable for live queries, validation, and demos, not historical modeling.